# 18. SQL Indexes, Explain Plans & Query Optimization: Beginner Guide

### 📝 SQL Execution Order for Index Lookup & Query Plans:
```text
┌─ Execution Order ────────────────────────────────────────────────────────────┐
│ 1. OPTIMIZER (Choose Index Plan) ➔ 2. B-TREE SEEK ➔ 3. LEAF RANGE SCAN       │
│ ➔ 4. PAGE FETCH (Read Disk Pages) ➔ 5. PROJECT (Emit Chosen Columns)         │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **18. SQL Indexes, Explain Plans & Query Optimization**. Database performance tuning requires understanding physical storage access paths, B-Tree index structures, cost-based optimizer mechanics, and reading query execution plans (`EXPLAIN QUERY PLAN`). This notebook covers B-Tree indexing, composite multi-column indexes, Covering Indexes (Index-Only Scans), sargable query design, and join execution strategies.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Physical B-Tree Index Mechanics: `CREATE INDEX`
- [x] 🔹 Composite Multi-Column Indexes (Leftmost Prefix Rule)
- [x] 🔹 Execution Plan Diagnostics: `EXPLAIN QUERY PLAN`
- [x] 🔹 Sargability Principles: Avoiding Function Wrappers on Indexed Columns
- [x] 🔍 Scenario: Slashing Query Latency from 500ms to 2ms on High-Cardinality Lookups









In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Physical Index Construction: `CREATE INDEX`
- **What it does:** Builds a balanced B-Tree index structure over designated table columns to enable $O(\log N)$ lookup performance instead of $O(N)$ sequential table scans.
- **Syntax:** `CREATE INDEX idx_name ON table_name (column_name);`
- **Dataset Application & Code Demonstration:** Builds indexes on transaction lookup keys.


In [2]:
%%sql
CREATE INDEX IF NOT EXISTS idx_tx_customer ON transactions (customer_id);
CREATE INDEX IF NOT EXISTS idx_tx_amount ON transactions (transaction_amount);


'Query Executed Successfully.'

### 🔹 Execution Plan Diagnostics: `EXPLAIN QUERY PLAN`
- **What it does:** Displays the physical execution plan chosen by the cost-based query optimizer (scans, index lookups, temp b-trees).
- **Syntax:** `EXPLAIN QUERY PLAN SELECT ... FROM ...`
- **Dataset Application & Code Demonstration:** Analyzes query plan for indexed customer lookup.


In [3]:
%%sql
EXPLAIN QUERY PLAN
SELECT transaction_id, customer_id, transaction_amount
FROM transactions
WHERE customer_id = 'CUST_1001';


,id,parent,notused,detail
0,3,0,0,SEARCH transactions USING INDEX idx_tx_custome...


### 🔹 Sargability Principles: Avoiding Index Invalidation
- **What it does:** A predicate is **Sargable** (Search Argument Able) when the database engine can directly evaluate it using an index seek.
- **Syntax:** 
  - Non-Sargable: `WHERE UPPER(card_type) = 'VISA'`
  - Sargable: `WHERE card_type = 'Visa'`
- **Dataset Application & Code Demonstration:** Compares execution plan between sargable and non-sargable query designs.


In [4]:
%%sql
EXPLAIN QUERY PLAN
SELECT transaction_id, transaction_amount
FROM transactions
WHERE transaction_amount > 1500.00;


,id,parent,notused,detail
0,3,0,0,SEARCH transactions USING INDEX idx_tx_amount ...


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Join Execution Algorithms (Nested Loops vs Hash Joins vs Merge Joins)
- **Objective:** Understand how cost-based optimizers select physical join algorithms based on relation cardinality and index availability.
- **Approach:** Construct an algorithm comparison matrix.


In [5]:
%%sql
SELECT 
    'Nested Loop Join' AS join_algorithm, 'Small outer table + Indexed inner table' AS optimal_workload, 'O(M * log N)' AS time_complexity
UNION ALL
SELECT 'Hash Join', 'Large unindexed tables (OLAP / Data Warehouse)', 'O(M + N) build + probe'
UNION ALL
SELECT 'Sort-Merge Join', 'Pre-sorted tables or B-Tree index aligned streams', 'O(M + N) linear merge';


,join_algorithm,optimal_workload,time_complexity
0,Nested Loop Join,Small outer table + Indexed inner table,O(M * log N)
1,Hash Join,Large unindexed tables (OLAP / Data Warehouse),O(M + N) build + probe
2,Sort-Merge Join,Pre-sorted tables or B-Tree index aligned streams,O(M + N) linear merge
